### Initialization

Run the following setup cells from the Decoders working directory in the qec environment. They load this notebook's configuration, paths and dependencies; subsequent experiments reuse these definitions and should be read in execution order.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
EXPERIMENT_DIR = ROOT / 'experiment/research/bp_failure_dynamics/stages/02_failure_mechanisms'
sys.path[:0] = [str(ROOT / 'src'), str(ROOT), str(ROOT / 'experiment')]

from experiment.utils.data_paths import ExperimentPaths

_data_paths = ExperimentPaths(ROOT, EXPERIMENT_DIR / 'config/paths.json')


# Next-stage BP failure mechanism experiments

This notebook classifies BP failure trajectories and tests controlled repair mechanisms. Figures are inline; per-input/per-iteration data go to `trace`, aggregates go to `stats`.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from itertools import combinations
from pathlib import Path
import json, sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import rankdata
from tqdm.auto import tqdm
from beliefmatching import detector_error_model_to_check_matrices
from codes.ca_code import CA_9_2_5, TriangularCode
from ldpc import BpDecoder
from noises_ca import CircuitBuilder, ErrorRate

CONFIG = json.loads((EXPERIMENT_DIR / 'config/bp_failure_mechanisms.json').read_text(encoding='utf-8'))
# Expected configuration values:
# print_level: integer; 1 = silent, 2 = key information, 3 = debug.
# Expected relationship: long_iterations >= benchmark_iterations >= diagnostic_iterations >= 3.
# ms_scaling_factor: positive.
# late_fraction: in (0, 1].
# primary_stability_threshold: chosen from stability_thresholds.
# stability_thresholds, residual_thresholds: each item is positive.
# fixed_patience_values, logical_stability_values: each item is positive.
# adaptive_k: non-negative.
# adaptive_max_views: non-negative.
# d9_weight4_bp_failure_samples_per_p: positive.
# d9_weight4_reuse_cached_bp_failures: boolean (True or False).
# equal_budget_runtime_repeats: at least 1.
# cross_distance_levels: [1, 2, 3].
# cross_distance_error_weights: values exactly {2, 3}.
# cross_distance_weight3_sample_limit: positive.
DATA = _data_paths.run()

from experiment.utils.misc import check_algorithm_cache
check_algorithm_cache(_data_paths, CONFIG, {'restart': 'gaussian_messages_v1'})
assert np.isfinite(CONFIG['restart_message_sigma']) and CONFIG['restart_message_sigma'] >= 0

from experiment.utils.bp_trace import BPContext, check_bp_cache
check_bp_cache(_data_paths, CONFIG)

INPUT = _data_paths.input('error_weight')
STATS = _data_paths.stats()
TRACE = _data_paths.trace()
STATS.mkdir(parents=True, exist_ok=True)
TRACE.mkdir(parents=True, exist_ok=True)

from experiment.utils import misc

def log(level, payload):
    return misc.log(level, payload, CONFIG=CONFIG)

sns.set_theme(style='whitegrid')
log(2, {'root': str(ROOT), 'stats': str(STATS), 'trace': str(TRACE)})


In [ ]:
DTYPE = {
    'p': float,
    'err': str,
    'det': str,
    'pred_b': str,
    'pred_t': str,
    'true': str,
}

weight2 = pd.read_csv(_data_paths.input('weight_2_data_csv'), dtype=DTYPE)
weight3 = pd.read_csv(_data_paths.input('weight_3_data_csv'), dtype=DTYPE)

required = set(DTYPE)
for frame, weight in ((weight2, 2), (weight3, 3)):
    assert required <= set(frame.columns) and frame.notna().all().all()
    assert frame.err.str.count('1').eq(weight).all()
    frame['syndrome_weight'] = frame.det.str.count('1').astype(int)

input_audit = pd.DataFrame([{
    'weight2_rows': len(weight2),
    'weight3_rows': len(weight3),
    'weight2_unique_decoder_inputs': len(weight2.drop_duplicates(['p', 'det', 'true'])),
    'weight3_unique_decoder_inputs': len(weight3.drop_duplicates(['p', 'det', 'true'])),
}])

input_audit.to_csv(_data_paths.output('next_stage_input_audit_csv'), index=False)
input_audit


## Instrumented BP and taxonomy helpers

Define the native BP trace interface, trajectory summaries and offline failure taxonomy. The classification uses logical truth for diagnosis; it is not itself an online stopping policy.


In [ ]:
from experiment.utils import bp_message_dynamics

Context = bp_message_dynamics.Context
MessageTrace = bp_message_dynamics.MessageTrace
CACHE = {}

def context_for(p):
    return bp_message_dynamics.context_for(
        p,
        CACHE=CACHE,
        CA_9_2_5=CA_9_2_5,
        CircuitBuilder=CircuitBuilder,
        Context=Context,
        ErrorRate=ErrorRate,
        detector_error_model_to_check_matrices=detector_error_model_to_check_matrices,
    )

from experiment.utils import misc

bit_array = misc.bit_array
bit_text = misc.bit_text
logical = bp_message_dynamics.logical

def trace_min_sum(
    context,
    syndrome,
    iterations,
    stop_early=False,
    *,
    rng=None,
    initial_message_sigma=0.0,
):
    return bp_message_dynamics.trace_min_sum(context, syndrome, iterations, stop_early, CONFIG=CONFIG, MessageTrace=MessageTrace, rng=rng, initial_message_sigma=initial_message_sigma)

def trace_prefix(trace, iterations):
    return bp_message_dynamics.trace_prefix(trace, iterations, MessageTrace=MessageTrace)

def benchmark_state(trace):
    return bp_message_dynamics.benchmark_state(trace, CONFIG=CONFIG)

def edge_metrics(context, trace):
    return bp_message_dynamics.edge_metrics(context, trace, CONFIG=CONFIG)

def variable_metrics(context, trace, edges):
    return bp_message_dynamics.variable_metrics(context, trace, edges, CONFIG=CONFIG)

classify_outcome = bp_message_dynamics.classify_outcome
METRICS = ['max_normalized_late_residual', 'mean_normalized_late_residual', 'max_flip_rate', 'mean_flip_rate', 'max_period2_strength', 'mean_message_instability', 'mean_belief_instability', 'mean_extrinsic_conflict']


In [ ]:
from collections import Counter, defaultdict
from time import perf_counter
import networkx as nx
from ldpc_modified import AdaptiveBpDecoder

SECTORS = ['00', '01', '10', '11']

def normalized_iteration_residuals(context, trace):
    scale = np.abs(context.prior_llr[context.edge_variables]) + np.finfo(float).eps
    return np.max(np.abs(np.diff(trace.c2v, axis=0)) / scale, axis=1)

from experiment.utils import bp_message_dynamics

def normalized_late_residual(context, trace, endpoint):
    return bp_message_dynamics.normalized_late_residual(context, trace, endpoint, CONFIG=CONFIG)

def logical_histories(context, trace, true_label):
    labels = np.asarray([bit_text(logical(context, hard)) for hard in trace.hard])
    correct = labels == bit_text(true_label)
    return (labels, correct)

def endpoint_type(valid, correct, stable):
    if not valid:
        return 'dynamic_syndrome_failure'
    if correct:
        return 'not_failure'

    return 'stable_logical_selection_failure' if stable else 'unstable_logical_failure'

def trajectory_type(
    benchmark_valid,
    benchmark_correct,
    valid,
    correct,
    benchmark_iteration,
):
    if not benchmark_valid:
        return 'not_applicable'
    if benchmark_correct:
        return 'not_failure'
    later = np.arange(len(valid)) > benchmark_iteration

    return 'premature_first_valid_failure' if np.any(later & valid & correct) else 'persistent_wrong_logical_sector'

def exclusive_type(endpoint, trajectory):
    if endpoint == 'dynamic_syndrome_failure':
        return endpoint
    if endpoint == 'not_failure':
        return 'not_failure'

    if trajectory == 'premature_first_valid_failure':
        return trajectory
    prefix = 'stable' if endpoint == 'stable_logical_selection_failure' else 'unstable'

    return prefix + '_persistent_logical_failure'

def choose_stopping_index(
    rule,
    valid,
    labels,
    residuals,
    first_valid,
    limit,
):
    if rule['kind'] == 'S0':
        return first_valid if first_valid is not None and first_valid <= CONFIG['benchmark_iterations'] else CONFIG['benchmark_iterations']
    if first_valid is None:
        return limit

    if rule['kind'] == 'S1':
        return min(limit, first_valid + rule['k'])
    if rule['kind'] == 'S2':
        k = rule['k']
        for t in range(k - 1, limit + 1):
            sl = slice(t - k + 1, t + 1)
            if valid[sl].all() and np.all(labels[sl] == labels[t]):
                return t

        return limit

    if rule['kind'] == 'S3':
        for t in range(max(1, first_valid), limit + 1):
            if valid[t] and residuals[t - 1] < rule['tau']:
                return t

        return limit

    if rule['kind'] == 'S4':
        for t in range(max(1, first_valid + rule['k']), limit + 1):
            if valid[t] and residuals[t - 1] < rule['tau']:
                return t

        return limit

    raise AssertionError(rule)

def stopping_choice(
    rule,
    trace,
    labels,
    residuals,
):
    valid = trace.syndrome_consistent
    limit = CONFIG['long_iterations']
    first = trace.first_valid_iteration
    t = choose_stopping_index(rule, valid, labels, residuals, first, limit)

    if rule['kind'] != 'S1' or first is None or (t == limit and first + rule['k'] > limit):
        return t
    window = np.arange(first, t + 1)
    valid_window = window[valid[window]]

    if rule['decision'] == 'window_final':
        return t
    if rule['decision'] == 'last_valid':
        return int(valid_window[-1])
    counts = Counter(labels[valid_window])
    maximum = max(counts.values())
    tied = {label for label, count in counts.items() if count == maximum}

    return int([i for i in valid_window if labels[i] in tied][-1])

RULES = [{'name': 'S0', 'kind': 'S0'}]
for k in CONFIG['fixed_patience_values']:
    for decision in ('window_final', 'last_valid', 'majority_valid_sector'):
        RULES.append({
            'name': f'S1_K{k}_{decision}',
            'kind': 'S1',
            'k': k,
            'decision': decision,
        })

for k in CONFIG['logical_stability_values']:
    RULES.append({'name': f'S2_K{k}', 'kind': 'S2', 'k': k})
for tau in CONFIG['residual_thresholds']:
    RULES.append({'name': f'S3_tau{tau:g}', 'kind': 'S3', 'tau': tau})

for k in CONFIG['fixed_patience_values']:
    for tau in CONFIG['residual_thresholds']:
        RULES.append({
            'name': f'S4_K{k}_tau{tau:g}',
            'kind': 'S4',
            'k': k,
            'tau': tau,
        })

assert len({rule['name'] for rule in RULES}) == len(RULES)


## Instrumented/native consistency validation

Run the native/instrumented consistency checks before collecting mechanism statistics. Later trajectory comparisons rely on the same PCM, priors and syndrome conventions.


In [ ]:
validation_pool = weight2.drop_duplicates(['p', 'det'])
assert len(validation_pool) >= CONFIG['validation_cases']
validation = validation_pool.sample(CONFIG['validation_cases'], random_state=CONFIG['random_seed'])
validation_rows = []

for row in tqdm(validation.itertuples(index=False), total=len(validation), desc='Validate native BP', disable=CONFIG['print_level'] == 1):
    context = context_for(row.p)
    syndrome = bit_array(row.det)
    trace = trace_min_sum(context, syndrome, CONFIG['benchmark_iterations'])
    native = BpDecoder(
        context.H,
        error_channel=context.priors,
        max_iter=CONFIG['benchmark_iterations'],
        bp_method='minimum_sum',
        ms_scaling_factor=CONFIG['ms_scaling_factor'],
        schedule='parallel',
        input_vector_type='syndrome',
    )
    native_hard = np.asarray(native.decode(syndrome), dtype=np.uint8)
    native_iteration = int(native.iter)
    assert 0 <= native_iteration <= CONFIG['benchmark_iterations']
    native_llr = np.asarray(native.log_prob_ratios)
    np.testing.assert_allclose(trace.llr[native_iteration], native_llr, rtol=CONFIG['native_validation_tolerance'], atol=CONFIG['native_validation_tolerance'])
    hard_difference = trace.hard[native_iteration] != native_hard

    if hard_difference.any():
        assert np.max(np.abs(trace.llr[native_iteration, hard_difference])) <= CONFIG['native_validation_tolerance']
        assert np.max(np.abs(native_llr[hard_difference])) <= CONFIG['native_validation_tolerance']

    if native.converge:
        np.testing.assert_array_equal(context.H @ native_hard % 2, syndrome)
    validation_rows.append({
        'p': row.p,
        'syndrome': row.det,
        'native_iterations': native_iteration,
        'instrumented_first_valid_iteration': trace.first_valid_iteration,
        'syndrome_consistent': bool(native.converge),
        'near_zero_hard_tie_count': int(hard_difference.sum()),
        'max_abs_posterior_difference': float(np.max(np.abs(trace.llr[native_iteration] - native_llr))),
    })

validation_frame = pd.DataFrame(validation_rows)
paired_iterations = validation_frame.dropna(subset=['instrumented_first_valid_iteration'])
max_iteration_difference = float(np.max(np.abs(paired_iterations.native_iterations - paired_iterations.instrumented_first_valid_iteration))) if len(paired_iterations) else np.nan
validation_summary = pd.DataFrame([{
    'cases': len(validation_frame),
    'native_syndrome_consistent_count': int(validation_frame.syndrome_consistent.sum()),
    'near_zero_hard_tie_count': int(validation_frame.near_zero_hard_tie_count.sum()),
    'max_abs_posterior_difference': validation_frame.max_abs_posterior_difference.max(),
    'max_abs_native_vs_first_valid_iteration_difference': max_iteration_difference,
}])

validation_summary.to_csv(_data_paths.output('next_stage_native_validation_summary_csv'), index=False)
validation_summary


## Experiments 1--4: taxonomy, trajectories, stopping, and fixed points

The next cells classify failures, inspect representative trajectories and compare alternative stopping times. The key distinction is whether a later consistent iterate becomes logically correct or the wrong sector persists.


In [ ]:
frames = []
for source, error_weight in ((weight2, 2), (weight3, 3)):
    frame = source.copy()
    frame['error_weight'] = error_weight
    frames.append(frame)

population = pd.concat(frames, ignore_index=True).sample(frac=1, random_state=CONFIG['random_seed']).reset_index(drop=True)
support_counts = pd.concat([weight2.assign(error_weight=2), weight3.assign(error_weight=3)]).groupby(['p', 'det', 'true', 'error_weight']).err.nunique()
population['physical_support_multiplicity'] = [support_counts.loc[r.p, r.det, r.true, r.error_weight] for r in population.itertuples()]
failure_rows, trajectory_rows, residual_rows, taxonomy_rows = ([], [], [], [])
rule_totals = {r['name']: {
    'iterations': [],
    'logical_errors': 0,
    'valid': 0,
    'decoder_failures': 0,
    'repaired_w2': 0,
    'repaired_w3': 0,
} for r in RULES}
rule_repairs = defaultdict(lambda: [0, 0])
transition = pd.DataFrame(0, index=SECTORS, columns=SECTORS, dtype=int)

for sample_id, row in enumerate(tqdm(population.itertuples(index=False), total=len(population), desc='Next-stage trajectories', disable=CONFIG['print_level'] == 1)):
    ctx = context_for(row.p)
    syndrome = bit_array(row.det)
    truth = bit_array(row.true)
    error = bit_array(row.err)
    trace = trace_min_sum(ctx, syndrome, CONFIG['long_iterations'])
    labels, correct = logical_histories(ctx, trace, truth)
    residuals = normalized_iteration_residuals(ctx, trace)
    first = trace.first_valid_iteration
    benchmark_t = first if first is not None and first <= CONFIG['benchmark_iterations'] else CONFIG['benchmark_iterations']
    benchmark_valid = bool(trace.syndrome_consistent[benchmark_t])
    benchmark_correct = bool(correct[benchmark_t])
    late = normalized_late_residual(ctx, trace, CONFIG['diagnostic_iterations'])
    primary_stable = late < CONFIG['primary_stability_threshold']
    ep = endpoint_type(benchmark_valid, benchmark_correct, primary_stable)
    tr_type = trajectory_type(benchmark_valid, benchmark_correct, trace.syndrome_consistent, correct, benchmark_t)
    ex_type = exclusive_type(ep, tr_type)
    baseline_failure = not (benchmark_valid and benchmark_correct)
    first_correct = np.flatnonzero(trace.syndrome_consistent & correct)
    logical_delay = np.nan if first is None or first_correct.size == 0 else int(first_correct[0] - first)

    if baseline_failure:
        failure_rows.append({
            'sample_id': sample_id,
            'p': row.p,
            'distance': 5,
            'error_weight': row.error_weight,
            'syndrome_weight': row.syndrome_weight,
            'benchmark_syndrome_consistent': benchmark_valid,
            'benchmark_logical_correct': benchmark_correct,
            'max_normalized_late_residual': late,
            'first_valid_iteration': first,
            'first_correct_valid_iteration': np.nan if first_correct.size == 0 else int(first_correct[0]),
            'logical_delay': logical_delay,
            'later_correct_valid_exists': bool(first_correct.size and first_correct[-1] > benchmark_t),
            'endpoint_failure_type': ep,
            'trajectory_failure_type': tr_type,
            'exclusive_failure_type': ex_type,
        })
        valid_count = int(trace.syndrome_consistent[1:].sum())
        valid_correct_count = int((trace.syndrome_consistent[1:] & correct[1:]).sum())
        logical_switches = int(np.sum(labels[1:] != labels[:-1]))
        valid_switches = int(np.sum(trace.syndrome_consistent[1:] & trace.syndrome_consistent[:-1] & (labels[1:] != labels[:-1])))
        failure_rows[-1].update({
            'logical_sector_switches': logical_switches,
            'valid_sector_switches': valid_switches,
            'correct_sector_occupancy': valid_correct_count / max(1, valid_count),
        })

        for t in range(CONFIG['long_iterations'] + 1):
            trajectory_rows.append({
                'sample_id': sample_id,
                'iteration': t,
                'syndrome_consistent': bool(trace.syndrome_consistent[t]),
                'logical_prediction': labels[t],
                'logical_correct': bool(correct[t]),
                'max_normalized_edge_residual': np.nan if t == 0 else residuals[t - 1],
            })

        for t in range(CONFIG['long_iterations']):
            if trace.syndrome_consistent[t] and trace.syndrome_consistent[t + 1]:
                transition.loc[labels[t], labels[t + 1]] += 1

    for rule in RULES:
        t = stopping_choice(rule, trace, labels, residuals)
        is_valid = bool(trace.syndrome_consistent[t])
        is_correct = bool(correct[t])
        repaired = baseline_failure and is_valid and is_correct
        stats = rule_totals[rule['name']]
        stats['iterations'].append(t)
        stats['logical_errors'] += int(not is_correct)
        stats['valid'] += int(is_valid)
        stats['decoder_failures'] += int(not (is_valid and is_correct))

        if repaired:
            stats[f'repaired_w{row.error_weight}'] += 1
        if baseline_failure:
            rule_repairs[rule['name'], ex_type][1] += 1
            rule_repairs[rule['name'], ex_type][0] += int(repaired)

    final = CONFIG['long_iterations']
    final_valid = bool(trace.syndrome_consistent[final])
    final_correct = bool(correct[final])
    final_late = normalized_late_residual(ctx, trace, final)

    if final_valid:
        correction = trace.hard[final]
        residual = error ^ correction
        assert not np.any(ctx.H @ residual % 2)

        if not final_correct:
            assert np.any(logical(ctx, residual))
        residual_rows.append({
            'sample_id': sample_id,
            'p': row.p,
            'distance': 5,
            'error_weight': row.error_weight,
            'syndrome_weight': row.syndrome_weight,
            'error_support': row.err,
            'residual_support': bit_text(residual),
            'final_logical_correct': final_correct,
            'final_stable': final_late < CONFIG['primary_stability_threshold'],
            'max_normalized_late_residual': final_late,
            'posterior_min_abs_llr': np.min(np.abs(trace.llr[final])),
            'posterior_mean_abs_llr': np.mean(np.abs(trace.llr[final])),
            'posterior_median_abs_llr': np.median(np.abs(trace.llr[final])),
            'correction_cost': float(ctx.prior_llr @ correction),
            'residual_weight': int(residual.sum()),
            'residual_cost': float(ctx.prior_llr @ residual),
            'exact_sampled_error_match': int(not residual.any()),
        })

    for tau in CONFIG['stability_thresholds']:
        taxonomy_rows.append({
            'sample_id': sample_id,
            'p': row.p,
            'distance': 5,
            'error_weight': row.error_weight,
            'syndrome_weight': row.syndrome_weight,
            'stability_threshold': tau,
            'benchmark_syndrome_consistent': benchmark_valid,
            'benchmark_logical_correct': benchmark_correct,
            'max_normalized_late_residual': late,
            'stable': late < tau,
            'first_valid_iteration': first,
            'later_correct_valid_exists': bool(first_correct.size and first_correct[-1] > benchmark_t),
            'failure_type': exclusive_type(endpoint_type(benchmark_valid, benchmark_correct, late < tau), tr_type),
        })

failures = pd.DataFrame(failure_rows)
trajectories = pd.DataFrame(trajectory_rows)
residual_metrics = pd.DataFrame(residual_rows)
taxonomy = pd.DataFrame(taxonomy_rows)
assert len(population) == len(weight2) + len(weight3) and len(failures) > 0 and (len(trajectories) == len(failures) * (CONFIG['long_iterations'] + 1))
assert failures.sample_id.is_unique and (not taxonomy.duplicated(['sample_id', 'stability_threshold']).any())

taxonomy.to_csv(_data_paths.output('failure_type_population_csv'), index=False)
failures.to_csv(_data_paths.output('failure_input_summary_csv'), index=False)
trajectories.to_csv(_data_paths.output('logical_sector_trajectories_csv'), index=False)
residual_metrics.to_csv(_data_paths.output('residual_operator_metrics_csv'), index=False)


In [ ]:
taxonomy_summary = taxonomy[taxonomy.failure_type != 'not_failure'].groupby(['stability_threshold', 'p', 'error_weight', 'failure_type'], as_index=False).size().rename(columns={'size': 'cases'})
taxonomy_summary['fraction_within_stratum'] = taxonomy_summary.cases / taxonomy_summary.groupby(['stability_threshold', 'p', 'error_weight']).cases.transform('sum')
taxonomy_summary.to_csv(_data_paths.output('failure_type_population_summary_csv'), index=False)
delay_summary = failures.groupby(['error_weight', 'exclusive_failure_type'], as_index=False).agg(
    cases=('sample_id', 'size'),
    later_correct_valid=('later_correct_valid_exists', 'sum'),
    logical_delay_count=('logical_delay', 'count'),
    logical_delay_mean=('logical_delay', 'mean'),
    logical_delay_median=('logical_delay', 'median'),
    mean_sector_switches=('logical_sector_switches', 'mean'),
    mean_correct_sector_occupancy=('correct_sector_occupancy', 'mean'),
)

delay_summary.to_csv(_data_paths.output('logical_delay_summary_csv'), index=False)
transition_long = transition.rename_axis('from_sector').reset_index().melt('from_sector', var_name='to_sector', value_name='count')
transition_long['probability'] = transition_long['count'] / transition_long.groupby('from_sector')['count'].transform('sum').replace(0, np.nan)

transition_long.to_csv(_data_paths.output('logical_sector_transition_matrix_csv'), index=False)
benchmark_rows = []
for name, stats in rule_totals.items():
    it = np.asarray(stats['iterations'])
    n = len(population)
    benchmark_rows.append({
        'rule': name,
        'cases': n,
        'logical_error_rate': stats['logical_errors'] / n,
        'syndrome_consistent_rate': stats['valid'] / n,
        'decoder_failure_rate': stats['decoder_failures'] / n,
        'mean_iterations': it.mean(),
        'median_iterations': np.median(it),
        'p95_iterations': np.quantile(it, 0.95),
        'max_iterations': it.max(),
        'weight2_failures_repaired': stats['repaired_w2'],
        'weight3_failures_repaired': stats['repaired_w3'],
    })

stopping_benchmark = pd.DataFrame(benchmark_rows)
stopping_benchmark.to_csv(_data_paths.output('stopping_rule_benchmark_csv'), index=False)
repair_rows = [{
    'rule': rule,
    'failure_type': failure_type,
    'repaired': values[0],
    'cases': values[1],
    'repair_fraction': values[0] / values[1],
} for (rule, failure_type), values in rule_repairs.items()]

pd.DataFrame(repair_rows).to_csv(_data_paths.output('stopping_rule_repair_by_failure_type_csv'), index=False)
stable = residual_metrics[residual_metrics.final_stable].copy()
fixed_metrics = ['posterior_min_abs_llr', 'posterior_mean_abs_llr', 'posterior_median_abs_llr', 'correction_cost', 'residual_weight', 'residual_cost', 'exact_sampled_error_match']
fixed_rows = []

for correct, group in stable.groupby('final_logical_correct'):
    for metric in fixed_metrics:
        x = group[metric]
        fixed_rows.append({
            'group': 'stable-valid-correct' if correct else 'stable-valid-logical-wrong',
            'metric': metric,
            'count': len(x),
            'mean': x.mean(),
            'median': x.median(),
            'std': x.std(),
            'q25': x.quantile(0.25),
            'q75': x.quantile(0.75),
        })

fixed_comparison = pd.DataFrame(fixed_rows)
fixed_comparison.to_csv(_data_paths.output('stable_fixed_point_comparison_csv'), index=False)


## Experiment 5: residual-operator Tanner structure

Stable wrong/correct endpoints are exactly matched on `(p, error_weight, syndrome_weight)`. Boundary and time-layer fields are not inferred because the DEM conversion exposes no reliable mapping for them.

Compare the residual between correction and sampled error for matched stable-wrong and stable-correct endpoints. Structural differences are evidence of association; they do not by themselves establish that a motif causes failure.


In [ ]:
def tanner_graph(context):
    graph = nx.Graph()
    graph.add_nodes_from([('v', v) for v in range(context.H.shape[1])], bipartite=0)
    graph.add_nodes_from([('c', c) for c in range(context.H.shape[0])], bipartite=1)
    graph.add_edges_from(((('c', int(c)), ('v', int(v))) for c, v in zip(context.edge_checks, context.edge_variables)))

    return graph

def neighborhood(graph, seeds, hops):
    nodes = set(seeds)
    for _ in range(hops):
        nodes |= {neighbor for node in tuple(nodes) for neighbor in graph.neighbors(node)}

    return graph.subgraph(nodes).copy()

def graph_values(graph):
    if graph.number_of_nodes() == 0:
        return {
            'nodes': 0,
            'edges': 0,
            'components': 0,
            'largest_component': 0,
            'cycle_rank': 0,
            'short_cycle_basis': 0,
            'density': 0.0,
        }

    components = list(nx.connected_components(graph))
    basis = nx.cycle_basis(graph)

    return {
        'nodes': graph.number_of_nodes(),
        'edges': graph.number_of_edges(),
        'components': len(components),
        'largest_component': max(map(len, components)),
        'cycle_rank': graph.number_of_edges() - graph.number_of_nodes() + len(components),
        'short_cycle_basis': sum((len(cycle) <= 8 for cycle in basis)),
        'density': nx.density(graph),
    }

wrong = stable[~stable.final_logical_correct].sample(frac=1, random_state=CONFIG['random_seed'])
correct = stable[stable.final_logical_correct].sample(frac=1, random_state=CONFIG['random_seed'] + 1)
pairs = []
pair_id = 0

for key, wgroup in wrong.groupby(['p', 'error_weight', 'syndrome_weight']):
    cgroup = correct[(correct.p == key[0]) & (correct.error_weight == key[1]) & (correct.syndrome_weight == key[2])]
    count = min(len(wgroup), len(cgroup))
    for (_, wr), (_, cr) in zip(wgroup.head(count).iterrows(), cgroup.head(count).iterrows()):
        pairs += [(pair_id, 'stable-valid-logical-wrong', wr), (pair_id, 'stable-valid-correct', cr)]
        pair_id += 1

structure_rows = []
for pair_id, group, row in tqdm(pairs, desc='Residual structure', disable=CONFIG['print_level'] == 1):
    ctx = context_for(row.p)
    full = tanner_graph(ctx)
    support = np.flatnonzero(bit_array(row.residual_support))
    seeds = {('v', int(v)) for v in support}
    values = {
        'matched_pair_id': pair_id,
        'sample_id': row.sample_id,
        'group': group,
        'p': row.p,
        'error_weight': row.error_weight,
        'syndrome_weight': row.syndrome_weight,
        'residual_weight': len(support),
        'observable_support_overlap': int(np.any(ctx.L[:, support], axis=0).sum()) if len(support) else 0,
    }

    for hops in (1, 2):
        values.update({f'h{hops}_{key}': value for key, value in graph_values(neighborhood(full, seeds, hops)).items()})
    structure_rows.append(values)

structure = pd.DataFrame(structure_rows)
assert len(structure) > 0 and structure.groupby('matched_pair_id').size().eq(2).all()
structure.to_csv(_data_paths.output('stable_structure_matched_cases_csv'), index=False)
structure_metrics = [c for c in structure if c.startswith(('h1_', 'h2_'))] + ['residual_weight', 'observable_support_overlap']
rows = []

for group, data in structure.groupby('group'):
    for metric in structure_metrics:
        rows.append({
            'group': group,
            'metric': metric,
            'count': len(data),
            'mean': data[metric].mean(),
            'median': data[metric].median(),
            'std': data[metric].std(),
        })

structure_summary = pd.DataFrame(rows)
structure_summary.to_csv(_data_paths.output('stable_wrong_structure_summary_csv'), index=False)
structure_summary


## Experiment 6: Adaptive BP repair by failure type

Apply Adaptive BP to the classified failures and report repairs by failure family. This asks which trajectories benefit from a changed graph, rather than attributing every improvement to suppression of oscillation.


In [ ]:
adaptive_rows = []
adaptive_cache = {}
for row in tqdm(failures.itertuples(index=False), total=len(failures), desc='Adaptive repair', disable=CONFIG['print_level'] == 1):
    ctx = context_for(row.p)
    syndrome = bit_array(population.loc[row.sample_id, 'det'])
    truth = bit_array(population.loc[row.sample_id, 'true'])

    if row.p not in adaptive_cache:
        adaptive_cache[row.p] = AdaptiveBpDecoder(
            ctx.H,
            error_channel=ctx.priors,
            max_iter=CONFIG['benchmark_iterations'],
            bp_method='minimum_sum',
            ms_scaling_factor=CONFIG['ms_scaling_factor'],
            schedule='parallel',
            input_vector_type='syndrome',
            adaptive_k=CONFIG['adaptive_k'],
            adaptive_max_views=CONFIG['adaptive_max_views'],
            view_thread_count=1,
        )
    decoder = adaptive_cache[row.p]
    runtimes = []

    for _ in range(CONFIG['equal_budget_runtime_repeats']):
        started = perf_counter()
        correction = np.asarray(decoder.decode(syndrome), dtype=np.uint8)
        runtimes.append(perf_counter() - started)

    valid = np.array_equal(ctx.H @ correction % 2, syndrome)
    correct = np.array_equal(logical(ctx, correction), truth)
    telemetry = decoder.adaptive_stats
    view_iterations = decoder.view_iterations.astype(int)
    view_edge_counts = decoder.view_edge_counts.astype(int)
    assert len(view_iterations) == len(view_edge_counts) == telemetry['bp_run_count']
    assert telemetry['bp_run_count'] == 1 + telemetry['executed_view_count']
    actual_updates = int(view_iterations @ view_edge_counts)
    configured_updates = int(CONFIG['benchmark_iterations'] * view_edge_counts.sum())
    continuation = residual_metrics.loc[residual_metrics.sample_id == row.sample_id, 'final_logical_correct']

    adaptive_rows.append({
        'sample_id': row.sample_id,
        'distance': 5,
        'error_weight': int(row.error_weight),
        'failure_type': row.exclusive_failure_type,
        'adaptive_repaired': int(valid and correct),
        'adaptive_syndrome_consistent': int(valid),
        'adaptive_logical_correct': int(correct),
        'plain_continuation_repaired': int(bool(continuation.iloc[0]) if len(continuation) else False),
        'adaptive_runtime_seconds': float(np.median(runtimes)),
        'adaptive_bp_run_count': int(telemetry['bp_run_count']),
        'adaptive_actual_edge_updates': actual_updates,
        'adaptive_configured_edge_updates': configured_updates,
        'adaptive_view_iterations': json.dumps(view_iterations.tolist()),
        'adaptive_view_edge_counts': json.dumps(view_edge_counts.tolist()),
        'adaptive_selected_rows_per_view': json.dumps(telemetry['selected_rows_per_view']),
        'adaptive_selected_row_weights_per_view': json.dumps(telemetry['selected_row_weights_per_view']),
    })

adaptive_cases = pd.DataFrame(adaptive_rows)
adaptive_cases.to_csv(_data_paths.output('adaptive_repair_cases_csv'), index=False)
adaptive_summary = adaptive_cases.groupby(['distance', 'error_weight', 'failure_type'], as_index=False).agg(
    num_cases=('sample_id', 'size'),
    adaptive_repaired=('adaptive_repaired', 'sum'),
    adaptive_syndrome_consistent=('adaptive_syndrome_consistent', 'sum'),
    plain_continuation_repaired=('plain_continuation_repaired', 'sum'),
    median_actual_edge_updates=('adaptive_actual_edge_updates', 'median'),
    median_runtime_seconds=('adaptive_runtime_seconds', 'median'),
)
adaptive_summary['repair_fraction'] = adaptive_summary.adaptive_repaired / adaptive_summary.num_cases
adaptive_summary['plain_continuation_repair_fraction'] = adaptive_summary.plain_continuation_repaired / adaptive_summary.num_cases

adaptive_summary.to_csv(_data_paths.output('adaptive_repair_by_failure_type_csv'), index=False)
adaptive_summary


In [ ]:
adaptive_summary = pd.read_csv(_data_paths.output('adaptive_repair_by_failure_type_csv'))
adaptive_summary


### d=9 weight-4 BP-failure sampling, taxonomy, and Adaptive BP repair

The cohort uses the `TriangularCode(3)` one-round circuit-level DEM with perfect measurement (`d_rounds=1`, `p_meas=0`). Physical weight-4 supports are scanned in a fixed-seed random order without replacement. Standard instrumented minimum-sum BP defines the baseline failure at the first syndrome-valid state within 60 iterations, or at iteration 60 when no valid state appears. Up to 1000 total BP failures are retained per physical error rate and then classified by the same four-class taxonomy as the distance-5 experiment. Decoder-input collisions are intentionally retained.

In [ ]:
from itertools import combinations
from time import perf_counter
from IPython.display import display
from ldpc_modified import AdaptiveBpDecoder

@dataclass(frozen=True)
class D9BpContext(BPContext):
    p: float
    H: np.ndarray
    L: np.ndarray
    priors: np.ndarray
    prior_llr: np.ndarray
    edge_checks: np.ndarray
    edge_variables: np.ndarray
    check_edges: tuple[np.ndarray, ...]
    variable_edges: tuple[np.ndarray, ...]

D9_BP_CONTEXT_CACHE = {}

def d9_bp_context(p):
    p = float(p)
    if p in D9_BP_CONTEXT_CACHE:
        return D9_BP_CONTEXT_CACHE[p]
    code = TriangularCode(3)
    circuit = CircuitBuilder(code).get_circuit_level_circuit(ErrorRate(p=p, p_meas=0), d_rounds=1)
    dem = circuit.detector_error_model()
    matrices = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=True)
    H = matrices.check_matrix.toarray().astype(np.uint8) % 2
    L = matrices.observables_matrix.toarray().astype(np.uint8) % 2
    priors = np.asarray(matrices.priors, dtype=float)
    checks, variables = np.nonzero(H)
    checks = checks.astype(np.int32)
    variables = variables.astype(np.int32)
    edge_ids = np.arange(len(checks), dtype=np.int32)
    context = D9BpContext(p, H, L, priors, np.log((1 - priors) / priors), checks, variables, tuple((edge_ids[checks == check] for check in range(H.shape[0]))), tuple((edge_ids[variables == variable] for variable in range(H.shape[1]))))
    assert code.d == 9
    assert H.shape == (dem.num_detectors, dem.num_errors)
    assert L.shape == (dem.num_observables, dem.num_errors)
    assert H.shape[1] == len(priors) and np.all((priors > 0) & (priors < 1))
    assert all((len(ids) >= 2 for ids in context.check_edges))
    result = (code, dem, context)
    D9_BP_CONTEXT_CACHE[p] = result

    return result

def d9_bits_text(bits):
    return ''.join((str(int(value)) for value in np.asarray(bits).reshape(-1)))

sample_path = _data_paths.output('d9_weight4_bp_failure_sample_csv')
audit_path = _data_paths.output('d9_weight4_bp_failure_sampling_audit_csv')
type_summary_path = _data_paths.output('d9_weight4_bp_failure_type_summary_csv')

d9_p_values = sorted((float(p) for p in pd.read_csv(_data_paths.input('weight_2_data_csv'), usecols=['p']).p.unique()))
target_per_p = int(CONFIG['d9_weight4_bp_failure_samples_per_p'])
required_sample_columns = {'sample_id', 'p', 'distance', 'error_weight', 'physical_support_rank', 'err', 'det', 'true', 'baseline_decoder', 'benchmark_iteration', 'benchmark_syndrome_consistent', 'benchmark_logical_correct', 'max_normalized_late_residual', 'endpoint_failure_type', 'trajectory_failure_type', 'failure_type'}
reuse_d9_sample = CONFIG['d9_weight4_reuse_cached_bp_failures'] and sample_path.exists()

if reuse_d9_sample:
    assert audit_path.exists(), audit_path
    d9_weight4_failures = pd.read_csv(sample_path, dtype={'err': str, 'det': str, 'true': str})
    d9_weight4_sampling_audit = pd.read_csv(audit_path)
    assert required_sample_columns.issubset(d9_weight4_failures.columns)
    assert set(d9_weight4_sampling_audit.target_failures_per_p) == {target_per_p}
    assert set(d9_weight4_sampling_audit.d_rounds) == {1}
    assert set(d9_weight4_sampling_audit.p_meas) == {0.0}
    assert set(d9_weight4_sampling_audit.bp_max_iter) == {CONFIG['benchmark_iterations']}
    assert set(d9_weight4_sampling_audit.trace_iterations) == {CONFIG['long_iterations']}
    assert set(d9_weight4_sampling_audit.diagnostic_iterations) == {CONFIG['diagnostic_iterations']}
    assert set(d9_weight4_sampling_audit.bp_method) == {'minimum_sum'}
    assert set(d9_weight4_sampling_audit.ms_scaling_factor) == {CONFIG['ms_scaling_factor']}
    assert set(d9_weight4_sampling_audit.stability_threshold) == {CONFIG['primary_stability_threshold']}
else:
    _, _, reference_context = d9_bp_context(d9_p_values[0])
    all_supports = np.asarray(list(combinations(range(reference_context.H.shape[1]), 4)), dtype=np.int16)
    total_supports = len(all_supports)
    assert total_supports > target_per_p
    failure_rows = []
    audit_rows = []
    next_sample_id = 0

    for p_index, p in enumerate(tqdm(d9_p_values, desc='d=9 weight-4 BP failure sampling', disable=CONFIG['print_level'] == 1)):
        _, _, context = d9_bp_context(p)
        assert context.H.shape == reference_context.H.shape
        rng = np.random.default_rng(CONFIG['random_seed'] + 9000 + p_index)
        candidate_order = rng.permutation(total_supports)
        retained = 0
        candidates_evaluated = 0

        for physical_support_rank in tqdm(
            candidate_order,
            total=total_supports,
            desc=f'p={p:.6g}',
            leave=False,
            disable=CONFIG['print_level'] == 1,
        ):
            support = all_supports[physical_support_rank]
            error = np.zeros(context.H.shape[1], dtype=np.uint8)
            error[support] = 1
            syndrome = context.H @ error % 2
            truth = context.L @ error % 2
            screening_trace = trace_min_sum(context, syndrome, CONFIG['benchmark_iterations'], stop_early=True)
            candidates_evaluated += 1
            screening_t = screening_trace.first_valid_iteration if screening_trace.first_valid_iteration is not None else CONFIG['benchmark_iterations']
            screening_valid = bool(screening_trace.syndrome_consistent[screening_t])
            screening_correct = np.array_equal(logical(context, screening_trace.hard[screening_t]), truth)

            if screening_valid and screening_correct:
                continue
            trace = trace_min_sum(context, syndrome, CONFIG['long_iterations'])
            labels, correct = logical_histories(context, trace, truth)
            first = trace.first_valid_iteration
            benchmark_t = first if first is not None and first <= CONFIG['benchmark_iterations'] else CONFIG['benchmark_iterations']
            benchmark_valid = bool(trace.syndrome_consistent[benchmark_t])
            benchmark_correct = bool(correct[benchmark_t])
            assert benchmark_valid == screening_valid
            assert benchmark_correct == screening_correct
            assert not (benchmark_valid and benchmark_correct)
            late = normalized_late_residual(context, trace, CONFIG['diagnostic_iterations'])
            endpoint = endpoint_type(benchmark_valid, benchmark_correct, late < CONFIG['primary_stability_threshold'])
            trajectory = trajectory_type(benchmark_valid, benchmark_correct, trace.syndrome_consistent, correct, benchmark_t)
            failure_type = exclusive_type(endpoint, trajectory)
            assert failure_type in {'dynamic_syndrome_failure', 'premature_first_valid_failure', 'stable_persistent_logical_failure', 'unstable_persistent_logical_failure'}
            first_correct = np.flatnonzero(trace.syndrome_consistent & correct)
            failure_rows.append({
                'sample_id': next_sample_id,
                'p': p,
                'distance': 9,
                'error_weight': 4,
                'physical_support_rank': int(physical_support_rank),
                'err': d9_bits_text(error),
                'det': d9_bits_text(syndrome),
                'true': d9_bits_text(truth),
                'syndrome_weight': int(syndrome.sum()),
                'baseline_decoder': 'BP',
                'benchmark_iteration': benchmark_t,
                'benchmark_syndrome_consistent': benchmark_valid,
                'benchmark_logical_correct': benchmark_correct,
                'first_valid_iteration': first,
                'first_correct_valid_iteration': np.nan if len(first_correct) == 0 else int(first_correct[0]),
                'later_correct_valid_exists': bool(np.any((np.arange(len(trace.syndrome_consistent)) > benchmark_t) & trace.syndrome_consistent & correct)),
                'max_normalized_late_residual': late,
                'stable': late < CONFIG['primary_stability_threshold'],
                'endpoint_failure_type': endpoint,
                'trajectory_failure_type': trajectory,
                'failure_type': failure_type,
            })
            next_sample_id += 1
            retained += 1

            if retained == target_per_p:
                break

        audit_rows.append({
            'p': p,
            'distance': 9,
            'error_weight': 4,
            'd_rounds': 1,
            'p_meas': 0.0,
            'bp_max_iter': CONFIG['benchmark_iterations'],
            'trace_iterations': CONFIG['long_iterations'],
            'diagnostic_iterations': CONFIG['diagnostic_iterations'],
            'bp_method': 'minimum_sum',
            'ms_scaling_factor': CONFIG['ms_scaling_factor'],
            'stability_threshold': CONFIG['primary_stability_threshold'],
            'physical_supports_total': total_supports,
            'candidates_evaluated': candidates_evaluated,
            'bp_failures_retained': retained,
            'target_failures_per_p': target_per_p,
            'target_reached': retained == target_per_p,
            'sampling_seed': CONFIG['random_seed'] + 9000 + p_index,
        })

    d9_weight4_failures = pd.DataFrame(failure_rows)
    d9_weight4_sampling_audit = pd.DataFrame(audit_rows)
    assert len(d9_weight4_failures) > 0
    decoder_input_counts = d9_weight4_failures.groupby('p', sort=True).apply(lambda frame: len(frame[['det', 'true']].drop_duplicates()), include_groups=False).rename('unique_decoder_inputs_retained')
    d9_weight4_sampling_audit = d9_weight4_sampling_audit.merge(decoder_input_counts, on='p', how='left', validate='one_to_one')

    d9_weight4_failures.to_csv(sample_path, index=False)
    d9_weight4_sampling_audit.to_csv(audit_path, index=False)

failure_types = {'dynamic_syndrome_failure', 'premature_first_valid_failure', 'stable_persistent_logical_failure', 'unstable_persistent_logical_failure'}
assert set(d9_weight4_sampling_audit.p) == set(d9_p_values)
assert d9_weight4_failures.sample_id.is_unique
assert not d9_weight4_failures.duplicated(['p', 'err']).any()
assert d9_weight4_failures.err.str.len().eq(56).all()
assert d9_weight4_failures.det.str.len().eq(24).all()
assert d9_weight4_failures.true.str.len().eq(2).all()
assert d9_weight4_failures.err.map(lambda text: text.count('1')).eq(4).all()
assert (~(d9_weight4_failures.benchmark_syndrome_consistent & d9_weight4_failures.benchmark_logical_correct)).all()
assert set(d9_weight4_failures.failure_type).issubset(failure_types)

for audit_row in d9_weight4_sampling_audit.itertuples(index=False):
    retained = int(np.isclose(d9_weight4_failures.p, audit_row.p).sum())
    assert retained == audit_row.bp_failures_retained <= target_per_p
    if audit_row.target_reached:
        assert retained == target_per_p
    else:
        assert audit_row.candidates_evaluated == audit_row.physical_supports_total

per_p_types = d9_weight4_failures.groupby(['p', 'failure_type'], as_index=False).agg(cases=('sample_id', 'size'))
per_p_types['scope'] = 'per_p'
per_p_types['fraction_within_scope'] = per_p_types.cases / per_p_types.groupby('p').cases.transform('sum')
all_p_types = d9_weight4_failures.groupby('failure_type', as_index=False).agg(cases=('sample_id', 'size'))
all_p_types.insert(0, 'p', np.nan)
all_p_types['scope'] = 'all_p'
all_p_types['fraction_within_scope'] = all_p_types.cases / all_p_types.cases.sum()
d9_weight4_failure_type_summary = pd.concat([per_p_types, all_p_types], ignore_index=True)[['scope', 'p', 'failure_type', 'cases', 'fraction_within_scope']]

d9_weight4_failure_type_summary.to_csv(type_summary_path, index=False)
adaptive_rows = []
adaptive_cache = {}

for row in tqdm(d9_weight4_failures.itertuples(index=False), total=len(d9_weight4_failures), desc='d=9 weight-4 Adaptive BP repair', disable=CONFIG['print_level'] == 1):
    _, _, context = d9_bp_context(row.p)
    syndrome = np.fromiter((ord(value) - 48 for value in row.det), dtype=np.uint8)
    truth = np.fromiter((ord(value) - 48 for value in row.true), dtype=np.uint8)
    key = float(row.p)

    if key not in adaptive_cache:
        adaptive_cache[key] = AdaptiveBpDecoder(
            context.H,
            error_channel=context.priors,
            max_iter=CONFIG['benchmark_iterations'],
            bp_method='minimum_sum',
            ms_scaling_factor=CONFIG['ms_scaling_factor'],
            schedule='parallel',
            input_vector_type='syndrome',
            adaptive_k=CONFIG['adaptive_k'],
            adaptive_max_views=CONFIG['adaptive_max_views'],
            view_thread_count=1,
        )
    decoder = adaptive_cache[key]
    runtimes = []

    for _ in range(CONFIG['equal_budget_runtime_repeats']):
        started = perf_counter()
        correction = np.asarray(decoder.decode(syndrome), dtype=np.uint8)
        runtimes.append(perf_counter() - started)

    syndrome_consistent = np.array_equal(context.H @ correction % 2, syndrome)
    logical_correct = np.array_equal(context.L @ correction % 2, truth)
    telemetry = decoder.adaptive_stats
    view_iterations = decoder.view_iterations.astype(int)
    view_edge_counts = decoder.view_edge_counts.astype(int)
    assert len(view_iterations) == len(view_edge_counts) == telemetry['bp_run_count']
    assert telemetry['bp_run_count'] == 1 + telemetry['executed_view_count']

    adaptive_rows.append({
        'sample_id': int(row.sample_id),
        'p': float(row.p),
        'distance': 9,
        'error_weight': 4,
        'baseline_decoder': 'BP',
        'failure_type': row.failure_type,
        'adaptive_repaired': int(syndrome_consistent and logical_correct),
        'adaptive_syndrome_consistent': int(syndrome_consistent),
        'adaptive_logical_correct': int(logical_correct),
        'adaptive_runtime_seconds': float(np.median(runtimes)),
        'adaptive_bp_run_count': int(telemetry['bp_run_count']),
        'adaptive_actual_edge_updates': int(view_iterations @ view_edge_counts),
        'adaptive_configured_edge_updates': int(CONFIG['benchmark_iterations'] * view_edge_counts.sum()),
        'adaptive_view_iterations': json.dumps(view_iterations.tolist()),
        'adaptive_view_edge_counts': json.dumps(view_edge_counts.tolist()),
        'adaptive_selected_rows_per_view': json.dumps(telemetry['selected_rows_per_view']),
        'adaptive_selected_row_weights_per_view': json.dumps(telemetry['selected_row_weights_per_view']),
    })

d9_weight4_adaptive_cases = pd.DataFrame(adaptive_rows)
assert len(d9_weight4_adaptive_cases) == len(d9_weight4_failures)
assert d9_weight4_adaptive_cases.sample_id.is_unique

d9_weight4_adaptive_cases.to_csv(_data_paths.output('d9_weight4_bp_adaptive_cases_csv'), index=False)
per_p_repair = d9_weight4_adaptive_cases.groupby(['p', 'failure_type'], as_index=False).agg(
    num_cases=('sample_id', 'size'),
    adaptive_repaired=('adaptive_repaired', 'sum'),
    adaptive_syndrome_consistent=('adaptive_syndrome_consistent', 'sum'),
    median_actual_edge_updates=('adaptive_actual_edge_updates', 'median'),
    median_runtime_seconds=('adaptive_runtime_seconds', 'median'),
)
per_p_repair['scope'] = 'per_p'
all_p_repair = d9_weight4_adaptive_cases.groupby('failure_type', as_index=False).agg(
    num_cases=('sample_id', 'size'),
    adaptive_repaired=('adaptive_repaired', 'sum'),
    adaptive_syndrome_consistent=('adaptive_syndrome_consistent', 'sum'),
    median_actual_edge_updates=('adaptive_actual_edge_updates', 'median'),
    median_runtime_seconds=('adaptive_runtime_seconds', 'median'),
)
all_p_repair.insert(0, 'p', np.nan)
all_p_repair['scope'] = 'all_p'
d9_weight4_adaptive_summary = pd.concat([per_p_repair, all_p_repair], ignore_index=True)
d9_weight4_adaptive_summary.insert(1, 'distance', 9)
d9_weight4_adaptive_summary.insert(2, 'error_weight', 4)
d9_weight4_adaptive_summary.insert(3, 'baseline_decoder', 'BP')
d9_weight4_adaptive_summary['repair_fraction'] = d9_weight4_adaptive_summary.adaptive_repaired / d9_weight4_adaptive_summary.num_cases

d9_weight4_adaptive_summary.to_csv(_data_paths.output('d9_weight4_bp_adaptive_repair_by_failure_type_csv'), index=False)
log(2, {
    'd9_weight4_bp_failure_cases': len(d9_weight4_failures),
    'all_p_targets_reached': bool(d9_weight4_sampling_audit.target_reached.all()),
})

display(d9_weight4_sampling_audit)
display(d9_weight4_failure_type_summary[d9_weight4_failure_type_summary.scope == 'all_p'])
d9_weight4_adaptive_summary[d9_weight4_adaptive_summary.scope == 'all_p']


## Experiment 7: equal-budget controls

The algorithmic budget is the measured sum of `iterations × Tanner-edge count` over BP views. Long/restart controls use the instrumented forced trajectory, while Adaptive and random-view runtimes use native BP; runtime values from different implementations are therefore reported but not treated as directly comparable.

Compare graph adaptation with longer plain BP, restarts and random redundant views under the recorded computation budget. These controls separate adaptive targeting from extra work and access to alternative graph representations.


In [ ]:
def correction_cost(context, correction):
    correction = np.asarray(correction, dtype=np.uint8)
    return float(np.sum(np.where(correction, -np.log(context.priors), -np.log1p(-context.priors))))

def choose_candidate(context, syndrome, candidates):
    candidates = [np.asarray(candidate, dtype=np.uint8) for candidate in candidates]
    valid = [candidate for candidate in candidates if np.array_equal(context.H @ candidate % 2, syndrome)]
    return min(valid, key=lambda candidate: correction_cost(context, candidate)) if valid else candidates[0]

def random_pair_pools(H):
    pools = defaultdict(list)
    for row_a, row_b in combinations(range(H.shape[0]), 2):
        redundant = H[row_a] ^ H[row_b]
        weight = int(redundant.sum())
        if weight:
            pools[weight].append((row_a, row_b, redundant))

    return pools

def build_random_views(
    context,
    syndrome,
    row_weights_per_view,
    rng,
):
    pools = random_pair_pools(context.H)
    used = {weight: set() for weight in pools}
    views = []

    for weights in row_weights_per_view:
        extra_rows = []
        extra_syndrome = []
        for weight in weights:
            assert weight in pools
            available = [index for index in range(len(pools[weight])) if index not in used[weight]]
            assert available
            selected = int(rng.choice(available))
            used[weight].add(selected)
            row_a, row_b, redundant = pools[weight][selected]
            extra_rows.append(redundant)
            extra_syndrome.append(int(syndrome[row_a] ^ syndrome[row_b]))

        assert extra_rows
        views.append((np.vstack([context.H, np.stack(extra_rows)]).astype(np.uint8), np.concatenate([syndrome, np.asarray(extra_syndrome, dtype=np.uint8)])))

    return views

equal_budget_rows = []
for row in tqdm(adaptive_cases.itertuples(index=False), total=len(adaptive_cases), desc='Equal-budget controls', disable=CONFIG['print_level'] == 1):
    failure = failures.loc[failures.sample_id == row.sample_id].iloc[0]
    ctx = context_for(failure.p)
    source = population.loc[row.sample_id]
    syndrome = bit_array(source.det)
    truth = bit_array(source.true)
    base_edges = int(ctx.H.sum())
    adaptive_budget = int(row.adaptive_actual_edge_updates)
    total_plain_iterations = max(1, adaptive_budget // base_edges)
    started = perf_counter()
    long_trace = trace_min_sum(ctx, syndrome, total_plain_iterations)
    long_runtime = perf_counter() - started
    long_candidate = long_trace.hard[-1]
    run_count = int(row.adaptive_bp_run_count)
    split = np.full(run_count, total_plain_iterations // run_count, dtype=int)
    split[:total_plain_iterations % run_count] += 1
    assert split.sum() == total_plain_iterations and np.all(split >= 1)
    restart_candidates = []
    started = perf_counter()

    restart_seeds = []
    for restart_id, iterations in enumerate(split):
        seed = int(np.random.SeedSequence([CONFIG['random_seed'], int(row.sample_id), restart_id]).generate_state(1)[0])
        restart_seeds.append(seed)
        restart_candidates.append(trace_min_sum(
            ctx, syndrome, int(iterations), rng=np.random.default_rng(seed),
            initial_message_sigma=CONFIG['restart_message_sigma'],
        ).hard[-1])
    restart_runtime = perf_counter() - started
    restart_candidate = choose_candidate(ctx, syndrome, restart_candidates)
    native_main = BpDecoder(
        ctx.H,
        error_channel=ctx.priors,
        max_iter=CONFIG['benchmark_iterations'],
        bp_method='minimum_sum',
        ms_scaling_factor=CONFIG['ms_scaling_factor'],
        schedule='parallel',
        input_vector_type='syndrome',
    )
    started = perf_counter()
    random_candidates = [np.asarray(native_main.decode(syndrome), dtype=np.uint8)]
    random_iterations = [int(native_main.iter)]
    random_edges = [base_edges]

    row_weights = json.loads(row.adaptive_selected_row_weights_per_view)
    rng = np.random.default_rng(CONFIG['random_view_seed'] + int(row.sample_id))
    for view_H, view_syndrome in build_random_views(ctx, syndrome, row_weights, rng):
        decoder = BpDecoder(
            view_H,
            error_channel=ctx.priors,
            max_iter=CONFIG['benchmark_iterations'],
            bp_method='minimum_sum',
            ms_scaling_factor=CONFIG['ms_scaling_factor'],
            schedule='parallel',
            input_vector_type='syndrome',
        )
        random_candidates.append(np.asarray(decoder.decode(view_syndrome), dtype=np.uint8))
        random_iterations.append(int(decoder.iter))
        random_edges.append(int(view_H.sum()))

    random_runtime = perf_counter() - started
    random_candidate = choose_candidate(ctx, syndrome, random_candidates)
    assert random_edges == json.loads(row.adaptive_view_edge_counts)

    variants = [('adaptive_bp', None, row.adaptive_runtime_seconds, row.adaptive_actual_edge_updates, row.adaptive_configured_edge_updates), ('longer_plain_bp', long_candidate, long_runtime, total_plain_iterations * base_edges, total_plain_iterations * base_edges), ('restarted_plain_bp', restart_candidate, restart_runtime, int(split.sum()) * base_edges, int(split.sum()) * base_edges), ('random_redundant_view_bp', random_candidate, random_runtime, int(np.asarray(random_iterations) @ np.asarray(random_edges)), CONFIG['benchmark_iterations'] * int(np.sum(random_edges)))]
    for variant, candidate, runtime, actual_updates, configured_updates in variants:
        if variant == 'adaptive_bp':
            valid = bool(row.adaptive_syndrome_consistent)
            correct = bool(row.adaptive_logical_correct)
        else:
            valid = np.array_equal(ctx.H @ candidate % 2, syndrome)
            correct = np.array_equal(logical(ctx, candidate), truth)

        equal_budget_rows.append({
            'sample_id': row.sample_id,
            'failure_type': row.failure_type,
            'variant': variant,
            'syndrome_consistent': int(valid),
            'logical_correct': int(correct),
            'repaired': int(valid and correct),
            'runtime_seconds': runtime,
            'actual_edge_updates': actual_updates,
            'configured_edge_updates': configured_updates,
            'actual_budget_ratio_to_adaptive': actual_updates / adaptive_budget,
            'restart_message_sigma': CONFIG['restart_message_sigma'] if variant == 'restarted_plain_bp' else np.nan,
            'restart_seeds_json': json.dumps(restart_seeds) if variant == 'restarted_plain_bp' else '[]',
        })

equal_budget_cases = pd.DataFrame(equal_budget_rows)
equal_budget_cases.to_csv(_data_paths.output('adaptive_equal_budget_cases_csv'), index=False)
equal_budget_comparison = equal_budget_cases.groupby('variant', as_index=False).agg(
    cases=('sample_id', 'size'),
    repaired=('repaired', 'sum'),
    syndrome_consistent=('syndrome_consistent', 'sum'),
    median_runtime_seconds=('runtime_seconds', 'median'),
    median_actual_edge_updates=('actual_edge_updates', 'median'),
    median_configured_edge_updates=('configured_edge_updates', 'median'),
    median_budget_ratio_to_adaptive=('actual_budget_ratio_to_adaptive', 'median'),
)
equal_budget_comparison['repair_fraction'] = equal_budget_comparison.repaired / equal_budget_comparison.cases
equal_budget_comparison['syndrome_consistent_fraction'] = equal_budget_comparison.syndrome_consistent / equal_budget_comparison.cases

equal_budget_comparison.to_csv(_data_paths.output('adaptive_equal_budget_comparison_csv'), index=False)
equal_budget_by_type = equal_budget_cases.groupby(['variant', 'failure_type'], as_index=False).agg(
    cases=('sample_id', 'size'),
    repaired=('repaired', 'sum'),
    median_actual_edge_updates=('actual_edge_updates', 'median'),
    median_runtime_seconds=('runtime_seconds', 'median'),
)
equal_budget_by_type['repair_fraction'] = equal_budget_by_type.repaired / equal_budget_by_type.cases

equal_budget_by_type.to_csv(_data_paths.output('adaptive_equal_budget_by_failure_type_csv'), index=False)
equal_budget_comparison


## Experiment 8: cross-distance scaling

Distance 5 reuses the exhaustive raw physical-support population above. Distances 3 and 9 are generated from the same one-round circuit-level construction. Distance-9 weight-3 physical supports are sampled with a fixed seed; the audit table records this sampling fraction.

Repeat the outcome classification at the configured distances while retaining each cohort's sampling procedure. A change across distances must be read together with the error-weight selection and achieved sample count.


In [ ]:
CROSS_CACHE = {}

def cross_context(level, p):
    key = (int(level), float(p))
    if key in CROSS_CACHE:
        return CROSS_CACHE[key]
    code = TriangularCode(level)
    circuit = CircuitBuilder(code).get_circuit_level_circuit(ErrorRate(p=float(p), p_meas=0), d_rounds=1)
    matrices = detector_error_model_to_check_matrices(circuit.detector_error_model(), allow_undecomposed_hyperedges=True)
    H = matrices.check_matrix.toarray().astype(np.uint8) % 2
    L = matrices.observables_matrix.toarray().astype(np.uint8) % 2
    priors = np.asarray(matrices.priors, dtype=float)
    checks, variables = np.nonzero(H)
    checks = checks.astype(np.int32)
    variables = variables.astype(np.int32)
    edge_ids = np.arange(len(checks), dtype=np.int32)
    context = Context(float(p), H, L, priors, np.log((1 - priors) / priors), checks, variables, tuple((edge_ids[checks == check] for check in range(H.shape[0]))), tuple((edge_ids[variables == variable] for variable in range(H.shape[1]))))
    assert code.d in {3, 5, 9} and H.shape[1] == len(priors)
    CROSS_CACHE[key] = (code.d, context)

    return CROSS_CACHE[key]

def selected_supports(num_variables, weight, level):
    supports = list(combinations(range(num_variables), weight))
    total = len(supports)
    limit = CONFIG['cross_distance_weight3_sample_limit'] if level == 3 and weight == 3 else total

    if limit < total:
        rng = np.random.default_rng(CONFIG['random_seed'] + 100 * level + weight)
        selected = np.sort(rng.choice(total, size=limit, replace=False))
        supports = [supports[index] for index in selected]

    return (supports, total)

cross_rows = []
cross_audit_rows = []
if CONFIG['cross_distance_reuse_distance5']:
    primary_taxonomy = taxonomy[np.isclose(taxonomy.stability_threshold, CONFIG['primary_stability_threshold'])].copy()
    d5 = primary_taxonomy.merge(failures[['sample_id', 'logical_delay']], on='sample_id', how='left', validate='one_to_one').merge(residual_metrics[['sample_id', 'final_stable', 'final_logical_correct', 'residual_weight']], on='sample_id', how='left', validate='one_to_one').merge(adaptive_cases[['sample_id', 'adaptive_repaired']], on='sample_id', how='left', validate='one_to_one')
    d5['adaptive_repaired'] = d5.adaptive_repaired.fillna(0).astype(int)
    d5['baseline_failure'] = d5.failure_type != 'not_failure'
    d5['distance'] = 5
    d5['sampled_physical_support'] = False
    cross_rows.extend(d5.to_dict('records'))

    for weight, source in ((2, weight2), (3, weight3)):
        total_per_p = len(source) // source.p.nunique()
        cross_audit_rows.append({
            'level': 2,
            'distance': 5,
            'error_weight': weight,
            'physical_supports_total': total_per_p,
            'physical_supports_selected': total_per_p,
            'sampling_fraction': 1.0,
            'reused_existing_population': True,
        })

p_values = sorted((float(p) for p in weight2.p.unique()))
for level in tqdm([value for value in CONFIG['cross_distance_levels'] if value != 2], desc='Cross-distance levels', disable=CONFIG['print_level'] == 1):
    distance, reference_context = cross_context(level, p_values[0])
    for error_weight in CONFIG['cross_distance_error_weights']:
        supports, total_supports = selected_supports(reference_context.H.shape[1], error_weight, level)
        cross_audit_rows.append({
            'level': level,
            'distance': distance,
            'error_weight': error_weight,
            'physical_supports_total': total_supports,
            'physical_supports_selected': len(supports),
            'sampling_fraction': len(supports) / total_supports,
            'reused_existing_population': False,
        })

        for p in tqdm(p_values, desc=f'd={distance}, w={error_weight}', leave=False, disable=CONFIG['print_level'] == 1):
            _, ctx = cross_context(level, p)
            identities = {}
            for support in supports:
                error = np.zeros(ctx.H.shape[1], dtype=np.uint8)
                error[list(support)] = 1
                syndrome = ctx.H @ error % 2
                truth = logical(ctx, error)
                identities.setdefault((bit_text(syndrome), bit_text(truth)), error)

            decoder = AdaptiveBpDecoder(
                ctx.H,
                error_channel=ctx.priors,
                max_iter=CONFIG['benchmark_iterations'],
                bp_method='minimum_sum',
                ms_scaling_factor=CONFIG['ms_scaling_factor'],
                schedule='parallel',
                input_vector_type='syndrome',
                adaptive_k=CONFIG['adaptive_k'],
                adaptive_max_views=CONFIG['adaptive_max_views'],
                view_thread_count=1,
            )
            for (syndrome_text, truth_text), error in identities.items():
                syndrome = bit_array(syndrome_text)
                truth = bit_array(truth_text)
                trace = trace_min_sum(ctx, syndrome, CONFIG['long_iterations'])
                labels, correct = logical_histories(ctx, trace, truth)
                first = trace.first_valid_iteration
                benchmark_t = first if first is not None and first <= CONFIG['benchmark_iterations'] else CONFIG['benchmark_iterations']
                benchmark_valid = bool(trace.syndrome_consistent[benchmark_t])
                benchmark_correct = bool(correct[benchmark_t])
                late = normalized_late_residual(ctx, trace, CONFIG['diagnostic_iterations'])
                endpoint = endpoint_type(benchmark_valid, benchmark_correct, late < CONFIG['primary_stability_threshold'])
                trajectory = trajectory_type(benchmark_valid, benchmark_correct, trace.syndrome_consistent, correct, benchmark_t)
                failure_type = exclusive_type(endpoint, trajectory)
                first_correct = np.flatnonzero(trace.syndrome_consistent & correct)
                logical_delay = np.nan if first is None or not len(first_correct) else int(first_correct[0] - first)
                final_stable = normalized_late_residual(ctx, trace, CONFIG['long_iterations']) < CONFIG['primary_stability_threshold']
                final_valid = bool(trace.syndrome_consistent[-1])
                final_correct = bool(correct[-1])
                residual_weight = np.nan

                if final_valid:
                    residual = error ^ trace.hard[-1]
                    assert not np.any(ctx.H @ residual % 2)
                    residual_weight = int(residual.sum())
                baseline_failure = not (benchmark_valid and benchmark_correct)
                adaptive_repaired = 0

                if baseline_failure:
                    adaptive_candidate = np.asarray(decoder.decode(syndrome), dtype=np.uint8)
                    adaptive_repaired = int(np.array_equal(ctx.H @ adaptive_candidate % 2, syndrome) and np.array_equal(logical(ctx, adaptive_candidate), truth))
                cross_rows.append({
                    'p': p,
                    'distance': distance,
                    'error_weight': error_weight,
                    'syndrome_weight': int(syndrome.sum()),
                    'benchmark_syndrome_consistent': benchmark_valid,
                    'benchmark_logical_correct': benchmark_correct,
                    'max_normalized_late_residual': late,
                    'stable': late < CONFIG['primary_stability_threshold'],
                    'first_valid_iteration': first,
                    'logical_delay': logical_delay,
                    'failure_type': failure_type,
                    'baseline_failure': baseline_failure,
                    'final_stable': final_stable,
                    'final_logical_correct': final_correct,
                    'residual_weight': residual_weight,
                    'adaptive_repaired': adaptive_repaired,
                    'sampled_physical_support': len(supports) < total_supports,
                })

cross_distance_cases = pd.DataFrame(cross_rows)
cross_distance_cases.insert(0, 'cross_sample_id', np.arange(len(cross_distance_cases)))
assert set(cross_distance_cases.distance) == {3, 5, 9}
assert (~cross_distance_cases.baseline_failure | (cross_distance_cases.failure_type != 'not_failure')).all()

cross_distance_cases.to_csv(_data_paths.output('cross_distance_cases_csv'), index=False)
cross_distance_audit = pd.DataFrame(cross_audit_rows)
cross_distance_audit.to_csv(_data_paths.output('cross_distance_input_audit_csv'), index=False)
failure_distance = cross_distance_cases[cross_distance_cases.baseline_failure].groupby(['distance', 'p', 'error_weight', 'failure_type'], as_index=False).size().rename(columns={'size': 'cases'})
failure_distance['fraction_within_stratum'] = failure_distance.cases / failure_distance.groupby(['distance', 'p', 'error_weight']).cases.transform('sum')

failure_distance.to_csv(_data_paths.output('failure_type_vs_distance_csv'), index=False)
cross_summary = cross_distance_cases.groupby(['distance', 'p'], as_index=False).agg(
    cases=('cross_sample_id', 'size'),
    bp_syndrome_consistent_rate=('benchmark_syndrome_consistent', 'mean'),
    bp_logical_error_rate=('benchmark_logical_correct', lambda values: 1 - values.mean()),
    decoder_failure_rate=('baseline_failure', 'mean'),
    stable_logical_wrong_fraction=('failure_type', lambda values: np.mean(values == 'stable_persistent_logical_failure')),
    premature_first_valid_fraction=('failure_type', lambda values: np.mean(values == 'premature_first_valid_failure')),
    median_logical_delay=('logical_delay', 'median'),
)
stable_wrong = cross_distance_cases[cross_distance_cases.final_stable & cross_distance_cases.final_logical_correct.eq(False)].groupby(['distance', 'p']).residual_weight.mean().rename('mean_stable_wrong_residual_weight')
cross_summary = cross_summary.merge(stable_wrong, on=['distance', 'p'], how='left')

cross_summary.to_csv(_data_paths.output('cross_distance_summary_csv'), index=False)
cross_adaptive = cross_distance_cases[cross_distance_cases.baseline_failure].groupby(['distance', 'failure_type'], as_index=False).agg(num_cases=('cross_sample_id', 'size'), adaptive_repaired=('adaptive_repaired', 'sum'))
cross_adaptive['repair_fraction'] = cross_adaptive.adaptive_repaired / cross_adaptive.num_cases

cross_adaptive.to_csv(_data_paths.output('cross_distance_adaptive_repair_csv'), index=False)
cross_summary


## Main figures

The next plotting cells summarize the taxonomy, timing and repair comparisons already computed above. Use the associated cohort sizes and controls when interpreting the visual differences.


In [ ]:
taxonomy_summary = pd.read_csv(_data_paths.output('failure_type_population_summary_csv'))
taxonomy_summary['failure_type'] = taxonomy_summary['failure_type'].replace('premature_first_valid_failure', 'premature_first_consistent_failure')
primary = taxonomy_summary[taxonomy_summary.stability_threshold == CONFIG['primary_stability_threshold']]

fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
df = primary.groupby(['p', 'failure_type']).cases.sum().unstack(fill_value=0).pipe(lambda x: x.div(x.sum(axis=1), axis=0))
df.plot.bar(stacked=True, ax=axes[0], title='Failure classes vs p')
axes[0].set_xticklabels([f'{p:.2e}' for p in df.index], rotation=45)

primary.groupby(['error_weight', 'failure_type']).cases.sum().unstack(fill_value=0).pipe(lambda x: x.div(x.sum(axis=1), axis=0)).plot.bar(stacked=True, ax=axes[1], title='Failure classes vs error weight')
plt.show()


In [ ]:
primary.groupby('failure_type').cases.sum().pipe(lambda x: x / x.sum())


In [ ]:
primary[primary['error_weight'] == 2]['failure_type'].value_counts()


In [ ]:
primary.groupby('error_weight').cases.sum()


In [ ]:
primary.groupby(['error_weight', 'failure_type']).cases.sum()


In [ ]:
primary.groupby(['error_weight', 'failure_type']).cases.sum().groupby(level='error_weight').transform(lambda x: x / x.sum())


In [ ]:
taxonomy_summary = pd.read_csv(_data_paths.output('failure_type_population_summary_csv'))
primary = taxonomy_summary[taxonomy_summary.stability_threshold == CONFIG['primary_stability_threshold']]
fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)
primary.groupby(['p', 'failure_type']).cases.sum().unstack(fill_value=0).pipe(lambda x: x.div(x.sum(axis=1), axis=0)).plot.bar(stacked=True, ax=axes[0, 0], title='Failure taxonomy vs p')
primary.groupby(['error_weight', 'failure_type']).cases.sum().unstack(fill_value=0).pipe(lambda x: x.div(x.sum(axis=1), axis=0)).plot.bar(stacked=True, ax=axes[0, 1], title='Failure taxonomy vs error weight')
failures.logical_delay.dropna().hist(ax=axes[1, 0])
axes[1, 0].set_title('Logical delay')

axes[1, 1].scatter(stopping_benchmark.mean_iterations, stopping_benchmark.decoder_failure_rate, s=15)
axes[1, 1].set(xlabel='mean iterations', ylabel='decoder failure rate', title='Stopping-rule tradeoff')
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
for ax, metric in zip(axes, ['posterior_min_abs_llr', 'correction_cost', 'residual_weight']):
    sns.boxplot(
        data=stable,
        x='final_logical_correct',
        y=metric,
        showfliers=False,
        ax=ax,
    )

plt.show()
transition_plot = transition_long.pivot(index='from_sector', columns='to_sector', values='probability').reindex(index=SECTORS, columns=SECTORS)
fig, ax = plt.subplots(figsize=(5, 4), constrained_layout=True)
sns.heatmap(
    transition_plot,
    annot=True,
    fmt='.3f',
    vmin=0,
    vmax=1,
    cmap='mako',
    ax=ax,
)
ax.set_title('Valid logical-sector transitions')

plt.show()
representatives = failures.groupby('exclusive_failure_type', sort=False).head(1).sample(frac=1, random_state=CONFIG['random_seed'])
fig, axes = plt.subplots(len(representatives), 1, figsize=(14, 2.8 * len(representatives)), sharex=True, constrained_layout=True)
axes = np.atleast_1d(axes)

for ax, row in zip(axes, representatives.itertuples(index=False)):
    case = trajectories[trajectories.sample_id == row.sample_id]
    sector_code = pd.Categorical(case.logical_prediction, categories=SECTORS).codes
    ax.step(case.iteration, sector_code, where='mid', label='logical sector')

    ax.scatter(case.loc[case.syndrome_consistent, 'iteration'], sector_code[case.syndrome_consistent], s=8, color='tab:green', label='syndrome valid')
    residual_axis = ax.twinx()
    residual_axis.plot(case.iteration, case.max_normalized_edge_residual, alpha=0.45, color='tab:red')
    residual_axis.set_yscale('log')
    ax.set_yticks(range(4), SECTORS)
    ax.set_title(row.exclusive_failure_type)

axes[-1].set_xlabel('iteration')
plt.show()
fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
sns.scatterplot(
    data=stable,
    x='correction_cost',
    y='posterior_min_abs_llr',
    hue='final_logical_correct',
    alpha=0.45,
    ax=ax,
)
ax.set_title('Stable fixed points')

plt.show()
example = stable[~stable.final_logical_correct].iloc[0]
example_context = context_for(example.p)
graph = tanner_graph(example_context)
active = {('v', int(index)) for index in np.flatnonzero(bit_array(example.residual_support))}
nodes = set(active)

for _ in range(2):
    nodes |= {neighbor for node in list(nodes) for neighbor in graph.neighbors(node)}
subgraph = graph.subgraph(nodes)
positions = nx.spring_layout(subgraph, seed=CONFIG['random_seed'])
colors = ['tab:red' if node in active else 'tab:blue' if node[0] == 'v' else 'tab:gray' for node in subgraph]

fig, ax = plt.subplots(figsize=(8, 6), constrained_layout=True)
nx.draw_networkx(
    subgraph,
    positions,
    node_color=colors,
    node_size=90,
    with_labels=False,
    ax=ax,
)
ax.set_title('Representative stable logical residual: two-hop Tanner neighborhood')

plt.show()
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
sns.barplot(
    data=adaptive_summary,
    x='failure_type',
    y='repair_fraction',
    ax=axes[0],
)
axes[0].tick_params(axis='x', rotation=35)
sns.scatterplot(
    data=equal_budget_comparison,
    x='median_actual_edge_updates',
    y='repair_fraction',
    hue='variant',
    s=90,
    ax=axes[1],
)
axes[0].set_title('Adaptive repair by failure type')
axes[1].set_title('Equal-budget controls')

plt.show()
fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)
distance_fraction = failure_distance.groupby(['distance', 'failure_type']).cases.sum().unstack(fill_value=0)

distance_fraction.div(distance_fraction.sum(axis=1), axis=0).plot.bar(stacked=True, ax=axes[0], title='Failure taxonomy vs distance')
sns.lineplot(
    data=cross_summary,
    x='distance',
    y='decoder_failure_rate',
    hue='p',
    marker='o',
    ax=axes[1],
)
sns.barplot(
    data=cross_adaptive,
    x='distance',
    y='repair_fraction',
    hue='failure_type',
    ax=axes[2],
)
axes[1].set_title('BP failure scaling')
axes[2].set_title('Adaptive repair scaling')

plt.show()


## Scope guardrails

Structural boundary/time-layer metrics remain omitted because DEM fault variables expose no reliable mapping. Distance-9 weight-3 results are fixed-seed sampled estimates, not exhaustive population values. Deterministic restart and forced-continuation controls isolate distinct mechanisms; their Python-instrumented wall-clock values are not directly comparable with native-decoder runtimes. Exact sampled-error equality is not decoding success.